# 02 · Evaluate RMSE vs SNR

Scores dictionary matching (DM) and three trained networks on the **same held-out test rows** at each SNR level.

| Method | Input | Training |
|---|---|---|
| **DM** | 40 L2-normalised echoes, matched to the noise-free dictionary | none |
| **DL-4p** | 40 L2-normalised echoes | mixed-SNR (checkpoint from `results/t2snr_results_v4`) |
| **Triple-NF** | 43-value input (echoes + R2* A/B/C) | noise-free |
| **Triple-Noisy** | 43-value input | mixed-SNR |

By default this uses the checkpoints shipped in `results/`, and **checks that the numbers reproduce**
`results/ablation_results/ablation_rmse.json`. The command-line equivalent is
`python scripts/run_pipeline.py evaluate --stats --compare-to results/ablation_results/ablation_rmse.json`.

Runtime is a few minutes on a GPU (dominated by dictionary matching).


In [ ]:
import logging, sys
from pathlib import Path

REPO = Path.cwd().resolve().parent          # this notebook lives in <repo>/notebooks
sys.path.insert(0, str(REPO / "src"))       # lets you run without `pip install -e .`

import matplotlib.pyplot as plt
from IPython.display import Image, display

from mrvf.config import load_config, with_overrides
from mrvf.training import get_device

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", datefmt="%H:%M:%S", force=True)


## Configuration

In [ ]:
# ── Edit these ────────────────────────────────────────────────────────────────────────
DATA_DIR   = REPO.parent / "subsamples" / "subsamples_v3"
ECHOTIMES  = REPO.parent / "echotimes.mat"
WITH_STATS = True    # also run paired Wilcoxon tests on the per-sample errors

# Checkpoints to evaluate. To score models you retrained in notebook 01, point these at runs/notebook_01/...
CHECKPOINTS = {
    "DL-4p":        REPO / "results/t2snr_results_v4/models/t2snr_noisy_4param_v4.pt",
    "Triple-NF":    REPO / "results/triple_regime_nf_results_v1/models/triple_nf_best.pt",
    "Triple-Noisy": REPO / "results/triple_regime_results_v1/models/triple_regime_best.pt",
}
REFERENCE = REPO / "results/ablation_results/ablation_rmse.json"   # what the published analysis saved
# ──────────────────────────────────────────────────────────────────────────────────────

from dataclasses import replace

cfg = load_config(REPO / "configs" / "default.toml")
cfg = replace(cfg, paths=replace(cfg.paths, dict_dir=DATA_DIR, echotimes_file=ECHOTIMES))
device = get_device()
print("device:", device)

## Run

In [ ]:
from mrvf.pipeline import run_evaluation

OUT = REPO / "runs" / "notebook_02"
out = run_evaluation(cfg, CHECKPOINTS, OUT, device, with_stats=WITH_STATS, make_figures=False)
results = out["results"]

## RMSE per method, parameter and SNR

In [ ]:
from mrvf.config import PARAM_KEYS, PARAM_LABELS, PARAM_UNITS

snrs = cfg.train.snr_levels
for key, label, unit in zip(PARAM_KEYS, PARAM_LABELS, PARAM_UNITS):
    print(f"\n{label} {unit}" + " " * 10 + "".join(f"SNR {s:<6}" for s in snrs))
    for method, curves in results.items():
        print(f"  {method:<14}" + "".join(f"{v:<10.3f}" for v in curves[key]))

In [ ]:
from mrvf import plotting

plotting.apply_style()
fig = plotting.plot_rmse_vs_snr(results, snrs, "RMSE vs SNR", OUT / "figures", "rmse_vs_snr")
plt.show()

## Does it reproduce the published numbers?

Compares every value with `results/ablation_results/ablation_rmse.json` (skipped if you evaluated different checkpoints).

In [ ]:
from mrvf.pipeline import compare_results

if REFERENCE.exists() and set(results) == {"DM", "DL-4p", "Triple-NF", "Triple-Noisy"}:
    diff = compare_results(results, REFERENCE)
    print(f"{diff['n_values']} values compared; max |difference| = {diff['max_abs_diff']:.3g}")
    assert diff["max_abs_diff"] < 1e-3, "results differ from the saved reference"
else:
    print("Reference comparison skipped (different set of methods).")

## Paired significance tests

One-sided Wilcoxon signed-rank test, per parameter and SNR, that each method's absolute error is larger than Triple-Noisy's.
Errors are used exactly as computed; no method's errors are rescaled. p-values are Bonferroni-adjusted across all comparisons.

In [ ]:
if WITH_STATS:
    stats = out["stats"]
    print(f"{'SNR':<5}{'method':<14}" + "".join(f"{p:<28}" for p in PARAM_KEYS))
    print(" " * 19 + "cells: mean |error| of Triple-Noisy → of the method, Bonferroni-adjusted p")
    for snr in snrs:
        for method, per_param in stats[snr].items():
            cells = [f"{per_param[p]['mean_ref']:.2f}→{per_param[p]['mean_other']:.2f}, p={per_param[p]['p_bonferroni']:.1g}" for p in PARAM_KEYS]
            print(f"{snr:<5}{method:<14}" + "".join(f"{c:<28}" for c in cells))